# QuantIQ | Phase 2 — Market Analysis

**Phase:** 2 | **Weeks:** 4–6 | **Deadline:** 14 June 2026

**Purpose:** Individual stock analysis for 12 NIFTY50 stocks — technical indicators, fundamentals, and cross-sector correlation.

---

## Team

| Handle | Stock          | Sector                |
|--------|----------------|-----------------------|
| AJ     | RELIANCE.NS    | Energy / Conglomerate |
| AV     | TCS.NS         | IT                    |
| AK     | INFY.NS        | IT                    |
| SS     | HCLTECH.NS     | IT                    |
| AR     | HDFCBANK.NS    | Banking               |
| EB     | ICICIBANK.NS   | Banking               |
| ShS    | AXISBANK.NS    | Banking               |
| RS     | TVSMOTOR.NS         | Auto                  |
| GT     | M&M.NS         | Auto                  |
| RT     | LT.NS          | Infrastructure        |
| NS     | TITAN.NS       | Consumer              |
| SmS    | APOLLOMICRO.NS | Defence               |

**Section 13 — Correlation Heatmap:** RS + GT

---

*Created by RS (Project Lead). Do not modify header cells or shared helper cells.*

In [107]:
import warnings
warnings.filterwarnings("ignore")

import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import yfinance as yf
from ta.trend import EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import AverageTrueRange
from src.data import fetch_ohlc

pio.templates["plotly_dark"].layout.paper_bgcolor = "#252526"
pio.templates["plotly_dark"].layout.plot_bgcolor  = "#1F2531"
pio.templates.default = "plotly_dark"

print("Imports OK")

Imports OK


In [108]:
TICKERS = {
    "AJ":  "RELIANCE.NS",
    "AV":  "TCS.NS",
    "AK":  "INFY.NS",
    "SS":  "HCLTECH.NS",
    "AR":  "HDFCBANK.NS",
    "EB":  "ICICIBANK.NS",
    "ShS": "AXISBANK.NS",
    "RS":  "TVSMOTOR.NS",
    "GT":  "M&M.NS",
    "RT":  "LT.NS",
    "NS":  "TITAN.NS",
    "SmS": "APOLLOMICRO.NS",
}

PERIOD   = "1y"
INTERVAL = "1d"

print(f"Config loaded — {len(TICKERS)} tickers, period={PERIOD}, interval={INTERVAL}")

Config loaded — 12 tickers, period=1y, interval=1d


In [109]:
def fetch_stock(
    ticker: str,
    period: str = PERIOD,
    interval: str = INTERVAL,
) -> pd.DataFrame:
    """Fetch OHLCV for a .NS ticker via fetch_ohlc.

    Args:
        ticker (str): NSE ticker with .NS suffix (e.g. "RELIANCE.NS").
        period (str): yfinance period string. Default "1y".
        interval (str): yfinance interval string. Default "1d".

    Returns:
        pd.DataFrame: OHLCV DataFrame indexed by datetime, NaN rows dropped.

    Raises:
        AssertionError: If ticker does not end with .NS.
        ValueError: If no data returned for ticker.
    """
    assert ticker.endswith(".NS"), f"Ticker must have .NS suffix, got: {ticker}"
    df = fetch_ohlc(ticker, period=period, interval=interval, use_cache=True)
    if df is None or df.empty:
        raise ValueError(f"No data returned for {ticker}")
    return df.dropna()

## Energy

## Section 1 — RELIANCE.NS | AJ (Energy / Conglomerate)

In [110]:
# ── 1a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AJ"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

RELIANCE.NS | 250 rows | 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-01,1326.367398,1329.353591,1312.431830,1313.924927,10700603
2026-06-02,1301.681536,1321.689127,1294.315690,1308.549805,23302785
2026-06-03,1308.948015,1317.906594,1295.012446,1307.156250,20012293
2026-06-04,1295.012426,1305.165434,1287.148760,1297.699951,23474327
2026-06-05,1304.500000,1306.000000,1288.000000,1291.000000,17785223


In [111]:
# ── 1b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,1424.63,1436.32,1412.68,1423.75,12982496.58
std,69.33,68.63,69.89,69.23,6876374.50
min,1289.04,1302.28,1284.06,1291.00,0.00
25%,1373.27,1382.61,1361.88,1371.31,8185545.75
50%,1413.46,1423.02,1398.93,1407.59,11128528.00
75%,1476.47,1485.73,1463.01,1472.83,16642564.50
max,1585.67,1604.38,1570.94,1584.97,42634247.00


In [112]:
# ── 1c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [113]:
# ── 1d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [114]:
# ── 1e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [115]:
# ── 1f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [116]:
# ── 1g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AJ"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 21.65
  EPS Growth (TTM) : -12.60%
  Debt / Equity    : 36.65
  Total Cash       : ₹2,581,470,117,888
  Total Debt       : ₹3,979,999,969,280
  Current Ratio    : 1.10
  PEG Ratio        : 0.82
  Dividend Yield   : 46.00%


### Observations

_TODO (AJ): Write 3–5 paragraphs on what the technical and fundamental data reveals about RELIANCE.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## IT

## Section 2 — TCS.NS | AV (IT)

In [117]:
# ── 2a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AV"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

TCS.NS | 250 rows | 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-01,2284.000000,2335.000000,2280.000000,2297.399902,7505095
2026-06-02,2320.000000,2457.399902,2316.199951,2446.899902,11306127
2026-06-03,2393.000000,2393.000000,2224.800049,2241.699951,15660062
2026-06-04,2241.699951,2253.100098,2216.600098,2241.000000,4867263
2026-06-05,2262.899902,2271.800049,2192.000000,2198.899902,5019717


In [118]:
# ── 2b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,2871.24,2893.17,2842.88,2865.29,3400395.06
std,319.98,316.88,320.95,320.71,2110665.64
min,2221.87,2234.70,2176.88,2198.90,0.00
25%,2553.70,2585.30,2526.96,2553.48,2270317.50
50%,2954.37,2977.40,2936.38,2956.12,2897469.50
75%,3109.04,3129.79,3083.69,3104.03,3904680.00
max,3385.87,3404.05,3356.04,3382.21,16331582.00


In [119]:
# ── 2c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [120]:
# ── 2d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [121]:
# ── 2e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [122]:
# ── 2f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [123]:
# ── 2g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AV"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 16.16
  EPS Growth (TTM) : 12.20%
  Debt / Equity    : 10.39
  Total Cash       : ₹410,799,996,928
  Total Debt       : ₹112,829,997,056
  Current Ratio    : 2.23
  PEG Ratio        : 2.43
  Dividend Yield   : 564.00%


### Observations

_TODO (AV): Write 3–5 paragraphs on what the technical and fundamental data reveals about TCS.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 3 — INFY.NS | AK (IT)

In [124]:
# ── 3a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AK"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

INFY.NS | 250 rows | 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-01,1175.000000,1217.000000,1173.699951,1202.500000,19439331
2026-06-02,1232.500000,1278.900024,1232.500000,1270.800049,36032600
2026-06-03,1242.099976,1249.900024,1216.000000,1222.599976,18534303
2026-06-04,1208.000000,1215.199951,1196.199951,1201.300049,12755957
2026-06-05,1220.000000,1223.800049,1194.300049,1197.500000,8692290


In [125]:
# ── 3b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,1450.06,1463.64,1435.06,1448.58,10081782.98
std,153.39,151.89,152.81,153.24,7249985.72
min,1102.00,1121.90,1089.00,1095.00,0.00
25%,1318.50,1327.47,1299.60,1313.65,6223763.75
50%,1485.46,1493.09,1472.02,1483.90,8190047.00
75%,1568.07,1585.93,1556.25,1577.80,12088294.75
max,1727.20,1728.00,1666.50,1689.80,69085430.00


In [126]:
# ── 3c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [127]:
# ── 3d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [128]:
# ── 3e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [129]:
# ── 3f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [130]:
# ── 3g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AK"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 15.58
  EPS Growth (TTM) : 11.80%
  Debt / Equity    : 9.83
  Total Cash       : ₹3,705,999,872
  Total Debt       : ₹967,000,000
  Current Ratio    : 1.98
  PEG Ratio        : 2.21
  Dividend Yield   : 418.00%


### Observations

_TODO (AK): Write 3–5 paragraphs on what the technical and fundamental data reveals about INFY.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 4 — HCLTECH.NS | SS (IT)

In [131]:
# ── 4a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["SS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

HCLTECH.NS | 250 rows | 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-01,1190.099976,1212.800049,1190.099976,1195.099976,5970435
2026-06-02,1210.000000,1257.000000,1202.199951,1243.500000,7663067
2026-06-03,1228.800049,1230.000000,1176.800049,1179.000000,4800922
2026-06-04,1166.400024,1176.500000,1158.199951,1168.300049,2025595
2026-06-05,1179.000000,1182.500000,1147.000000,1154.699951,1971395


In [132]:
# ── 4b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,1458.24,1472.23,1442.47,1456.64,3185349.53
std,149.19,148.60,148.11,149.59,2728440.64
min,1121.90,1143.20,1103.40,1124.00,0.00
25%,1371.00,1381.66,1347.46,1364.79,1931098.25
50%,1448.70,1459.13,1436.08,1445.27,2566080.00
75%,1589.53,1610.73,1580.25,1594.84,3456803.25
max,1746.56,1746.66,1668.56,1697.11,33066256.00


In [133]:
# ── 4c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [134]:
# ── 4d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [135]:
# ── 4e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [136]:
# ── 4f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [137]:
# ── 4g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["SS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 18.83
  EPS Growth (TTM) : -0.20%
  Debt / Equity    : 6.93
  Total Cash       : ₹3,204,999,936
  Total Debt       : ₹550,000,000
  Current Ratio    : 2.22
  PEG Ratio        : 2.39
  Dividend Yield   : 831.00%


### Observations

_TODO (SS): Write 3–5 paragraphs on what the technical and fundamental data reveals about HCLTECH.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Banking

## Section 5 — HDFCBANK.NS | AR (Banking)

In [138]:
# ── 5a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AR"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

HDFCBANK.NS | 250 rows | 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-01,749.000000,752.349976,739.200012,742.700012,47996441
2026-06-02,737.000000,753.599976,733.150024,748.250000,47314447
2026-06-03,744.450012,756.900024,742.599976,753.650024,36109480
2026-06-04,749.150024,757.299988,745.000000,754.200012,42673538
2026-06-05,753.950012,758.700012,744.650024,747.049988,22116568


In [139]:
# ── 5b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,2.500000e+02
mean,924.63,932.37,918.10,924.97,2.705422e+07
std,87.80,86.35,88.14,87.17,1.861607e+07
min,730.00,751.00,726.65,731.55,0.000000e+00
25%,869.50,879.82,857.20,870.93,1.517161e+07
50%,962.35,969.47,956.71,963.80,2.218912e+07
75%,991.23,998.51,985.95,992.00,3.457548e+07
max,1017.50,1020.50,1008.50,1012.90,1.716374e+08


In [140]:
# ── 5c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [141]:
# ── 5d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [142]:
# ── 5e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [143]:
# ── 5f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [144]:
# ── 5g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AR"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 16.67
  EPS Growth (TTM) : 7.50%
  Debt / Equity    : N/A
  Total Cash       : ₹3,119,260,631,040
  Total Debt       : ₹5,884,845,490,176
  Current Ratio    : N/A
  PEG Ratio        : 0.89
  Dividend Yield   : 174.00%


### Observations

_TODO (AR): Write 3–5 paragraphs on what the technical and fundamental data reveals about HDFCBANK.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 6 — ICICIBANK.NS | EB (Banking)

In [145]:
# ── 6a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["EB"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

ICICIBANK.NS | 250 rows | 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-01,1257.099976,1261.900024,1235.900024,1239.699951,9911975
2026-06-02,1233.000000,1237.699951,1220.699951,1226.599976,17920892
2026-06-03,1219.900024,1248.300049,1213.699951,1242.000000,13934754
2026-06-04,1232.199951,1262.300049,1232.199951,1251.699951,19461560
2026-06-05,1257.000000,1265.000000,1250.000000,1262.099976,10166393


In [146]:
# ── 6b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,1366.31,1376.89,1356.78,1366.91,12841996.08
std,63.84,62.11,64.34,63.39,6364639.93
min,1196.00,1222.10,1187.60,1205.90,0.00
25%,1341.02,1354.10,1335.77,1345.35,7810495.75
50%,1378.45,1387.75,1371.05,1379.85,11540415.00
75%,1412.00,1419.50,1403.05,1413.17,17010082.00
max,1488.51,1488.51,1466.78,1477.20,38929053.00


In [147]:
# ── 6c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [148]:
# ── 6d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [149]:
# ── 6e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [150]:
# ── 6f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [151]:
# ── 6g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["EB"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 16.89
  EPS Growth (TTM) : 8.40%
  Debt / Equity    : N/A
  Total Cash       : ₹2,649,807,126,528
  Total Debt       : ₹2,202,642,677,760
  Current Ratio    : N/A
  PEG Ratio        : 0.53
  Dividend Yield   : 87.00%


### Observations

_TODO (EB): Write 3–5 paragraphs on what the technical and fundamental data reveals about ICICIBANK.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 7 — AXISBANK.NS | ShS (Banking)

In [152]:
# ── 7a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["ShS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

AXISBANK.NS | 250 rows | 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-01,1289.000000,1293.199951,1267.000000,1275.900024,4311076
2026-06-02,1263.000000,1275.800049,1244.900024,1251.099976,11884695
2026-06-03,1245.099976,1264.800049,1230.300049,1255.199951,7953326
2026-06-04,1246.000000,1259.900024,1244.099976,1253.300049,6392593
2026-06-05,1260.000000,1276.000000,1252.099976,1272.300049,7825303


In [153]:
# ── 7b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,1226.78,1238.54,1215.75,1227.08,6733291.82
std,90.36,91.59,88.16,89.98,3993119.31
min,1047.60,1058.00,1042.50,1045.20,0.00
25%,1174.00,1179.99,1162.48,1170.65,4318245.75
50%,1234.50,1245.67,1224.45,1234.10,5836335.00
75%,1288.60,1298.53,1272.50,1286.47,8138919.50
max,1403.00,1418.30,1387.60,1403.00,38053695.00


In [154]:
# ── 7c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [155]:
# ── 7d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [156]:
# ── 7e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [157]:
# ── 7f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [158]:
# ── 7g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["ShS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 15.06
  EPS Growth (TTM) : 6.40%
  Debt / Equity    : N/A
  Total Cash       : ₹1,093,515,673,600
  Total Debt       : ₹2,805,106,212,864
  Current Ratio    : N/A
  PEG Ratio        : 0.54
  Dividend Yield   : 8.00%


### Observations

_TODO (ShS): Write 3–5 paragraphs on what the technical and fundamental data reveals about AXISBANK.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Auto

## Section 8 — TVSMOTOR.NS | RS (Auto)

In [159]:
# ── 8a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["RS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

TVSMOTOR.NS | 250 rows | 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-01,3382.0,3420.000000,3335.100098,3344.399902,1210591
2026-06-02,3328.0,3383.000000,3291.600098,3366.899902,516666
2026-06-03,3360.0,3366.800049,3301.399902,3316.899902,846948
2026-06-04,3310.0,3400.699951,3292.199951,3362.500000,1089659
2026-06-05,3380.0,3397.000000,3345.000000,3383.899902,813580


In [160]:
# ── 8b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,3415.32,3450.86,3373.21,3413.14,847990.06
std,327.37,329.99,321.37,325.81,538957.36
min,2695.58,2738.23,2645.85,2708.13,0.00
25%,3312.08,3369.57,3269.60,3313.92,516995.00
50%,3490.45,3519.35,3452.93,3487.21,730499.50
75%,3645.29,3673.53,3592.14,3640.77,1024891.50
max,3940.43,3956.17,3904.45,3940.43,3748828.00


In [161]:
# ── 8c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [162]:
# ── 8d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [163]:
# ── 8e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [164]:
# ── 8f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [165]:
# ── 8g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["RS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 53.19
  EPS Growth (TTM) : 19.10%
  Debt / Equity    : 306.09
  Total Cash       : ₹50,211,201,024
  Total Debt       : ₹327,910,096,896
  Current Ratio    : 0.98
  PEG Ratio        : N/A
  Dividend Yield   : 35.00%


### Observations

_TODO (RS): Write 3–5 paragraphs on what the technical and fundamental data reveals about TVSMOTOR.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 9 — M&M.NS | GT (Auto)

In [166]:
# ── 9a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["GT"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

M&M.NS | 250 rows | 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-01,3050.000000,3068.899902,2961.000000,2970.199951,5763247
2026-06-02,2913.100098,3008.699951,2913.100098,2998.300049,4045354
2026-06-03,2996.000000,3029.500000,2965.000000,3011.100098,3110137
2026-06-04,2985.000000,3067.100098,2974.199951,3016.100098,2788572
2026-06-05,3039.899902,3059.000000,3012.699951,3040.500000,1685633


In [167]:
# ── 9b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,3373.25,3409.87,3336.72,3371.66,2575013.94
std,246.52,241.98,246.17,242.95,1427131.87
min,2902.00,2994.44,2896.00,2931.10,0.00
25%,3150.22,3192.09,3124.39,3158.18,1702055.25
50%,3377.00,3417.00,3335.15,3383.75,2249328.50
75%,3608.15,3639.52,3577.65,3604.20,3043433.50
max,3805.90,3839.90,3785.00,3802.40,11402867.00


In [168]:
# ── 9c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [169]:
# ── 9d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [170]:
# ── 9e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [171]:
# ── 9f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [172]:
# ── 9g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["GT"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 19.96
  EPS Growth (TTM) : 44.80%
  Debt / Equity    : 125.07
  Total Cash       : ₹527,834,513,408
  Total Debt       : ₹1,369,359,122,432
  Current Ratio    : 1.27
  PEG Ratio        : 1.26
  Dividend Yield   : 109.00%


### Section 9 Technical & Fundamental Observations — M&M.NS

Mahindra & Mahindra (**M&M.NS**) exhibits a strong macro trend over the analyzed period, peaking significantly near the start of 2026 before entering a structural correction phase down toward the 3,000 baseline. On the technical side, the MACD indicator confirms this consolidation pattern with a clean crossover tracking the price correction, while the RSI (14) shows excellent resilience, repeatedly bouncing off the oversold 30 boundary line without breaking support. Fundamentally, M&M displays robust growth and valuation characteristics with a trailing P/E Ratio of 19.96 and strong EPS Growth (TTM) of 44.80%. Structurally, the firm maintains exceptional liquidity with a massive cash position exceeding ₹527 Billion against a manageable debt profile, cementing its healthy fundamental position relative to the broader Auto sector context.

## Infrastructure

## Section 10 — LT.NS | RT (Infrastructure)

In [173]:
# ── 10a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["RT"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

LT.NS | 250 rows | 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-01,4089.000000,4105.799805,4002.100098,4010.800049,1409388
2026-06-02,3980.000000,4017.000000,3923.500000,4000.899902,2085054
2026-06-03,4000.899902,4008.000000,3894.000000,3953.199951,1872929
2026-06-04,3945.000000,3974.899902,3930.100098,3942.100098,1398297
2026-06-05,3960.000000,3978.000000,3923.899902,3953.199951,1278243


In [174]:
# ── 10b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,3806.50,3838.26,3770.50,3803.75,2162864.14
std,225.94,227.87,225.82,227.92,1547452.27
min,3372.16,3376.92,3256.29,3310.07,0.00
25%,3596.37,3632.02,3562.97,3594.19,1243804.25
50%,3838.01,3881.44,3807.31,3841.28,1748995.50
75%,3992.50,4026.03,3961.48,3992.06,2334359.25
max,4372.79,4397.05,4329.41,4375.36,10778730.00


In [175]:
# ── 10c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [176]:
# ── 10d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [177]:
# ── 10e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [178]:
# ── 10f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [179]:
# ── 10g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["RT"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 33.79
  EPS Growth (TTM) : -34.50%
  Debt / Equity    : 97.64
  Total Cash       : ₹766,146,510,848
  Total Debt       : ₹1,254,965,903,360
  Current Ratio    : 1.25
  PEG Ratio        : 1.06
  Dividend Yield   : 96.00%


### Observations

_TODO (RT): Write 3–5 paragraphs on what the technical and fundamental data reveals about LT.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Consumer

## Section 11 — TITAN.NS | NS (Consumer)

In [180]:
# ── 11a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["NS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

TITAN.NS | 250 rows | 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-01,4100.000000,4105.000000,4007.000000,4024.600098,450200
2026-06-02,4017.399902,4088.000000,3986.000000,4078.100098,522015
2026-06-03,4057.000000,4103.799805,4002.000000,4088.800049,959352
2026-06-04,4075.000000,4273.799805,4052.300049,4231.000000,1761933
2026-06-05,4269.000000,4289.000000,4223.200195,4260.200195,1390200


In [181]:
# ── 11b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,3868.44,3906.31,3831.13,3872.15,994318.70
std,332.06,339.05,323.49,332.99,849697.20
min,3315.00,3347.30,3303.10,3316.00,0.00
25%,3552.60,3576.70,3517.08,3555.30,567838.75
50%,3872.20,3909.55,3843.05,3879.70,769350.50
75%,4137.50,4173.60,4082.08,4140.82,1107884.25
max,4554.00,4605.00,4467.70,4525.90,7524068.00


In [182]:
# ── 11c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [183]:
# ── 11d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [184]:
# ── 11e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [185]:
# ── 11f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [186]:
# ── 11g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["NS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 74.62
  EPS Growth (TTM) : 35.10%
  Debt / Equity    : 195.00
  Total Cash       : ₹41,659,998,208
  Total Debt       : ₹306,210,013,184
  Current Ratio    : 1.28
  PEG Ratio        : N/A
  Dividend Yield   : 26.00%


### Observations

_TODO (NS): Write 3–5 paragraphs on what the technical and fundamental data reveals about TITAN.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Defence

## Section 12 — APOLLOMICRO.NS | SmS (Defence)

In [187]:
# ── 12a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["SmS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: APOLLOMICRO.NS"}}}
$APOLLOMICRO.NS: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['APOLLOMICRO.NS']: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
yfinance returned empty DataFrame for APOLLOMICRO.NS (period=1y)


ValueError: No data returned for APOLLOMICRO.NS

In [ ]:
# ── 12b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 12c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 12d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 12e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 12f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 12g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["SmS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (SmS): Write 3–5 paragraphs on what the technical and fundamental data reveals about APOLLOMICRO.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 13 — Cross-Sector Correlation Heatmap | RS + GT

In [ ]:
# ── Section 13: Daily Return Correlation — RS + GT ──────────────────────────
# Uncomment and run after all 12 section data cells have been executed.

# from src.data import fetch_batch
#
# batch    = fetch_batch(list(TICKERS.values()), period=PERIOD, interval=INTERVAL)
# closes   = {
#     handle: batch[ticker]["Close"]
#     for handle, ticker in TICKERS.items()
#     if batch.get(ticker) is not None
# }
# close_df = pd.DataFrame(closes).dropna()
# returns  = close_df.pct_change().dropna()
# corr     = returns.corr()
#
# fig = px.imshow(
#     corr,
#     text_auto=".2f",
#     color_continuous_scale="RdBu_r",
#     zmin=-1, zmax=1,
#     title="NIFTY50 Sample — Daily Return Correlation (1Y)",
# )
# fig.update_layout(height=600)
# fig.show()

### Observations

_TODO (RS + GT): Write 3–5 paragraphs on correlation patterns — which sectors move together, which are uncorrelated, and what this implies for portfolio diversification._